In [1]:
library(dplyr)
library(tidyr)
library(purrr)
library(stringr)


source("NewSiteDataLoader.R")
source("data_loader.R")
source("meta.R")
source("recalMNE.R")
ls()


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



 [1] "accumulation_cause"                 "add_bone_info"                     
 [3] "add_element_count"                  "age_category"                      
 [5] "category_hierarchy"                 "category_mappings"                 
 [7] "category_mappings_arms_2_to_1"      "category_mappings_arms_3_to_1"     
 [9] "category_mappings_arms_3_to_2"      "category_mappings_foot_2_to_1"     
[11] "category_mappings_foot_3_to_1"      "category_mappings_foot_3_to_2"     
[13] "category_mappings_foot_4_to_1"      "category_mappings_foot_4_to_2"     
[15] "category_mappings_foot_4_to_3"      "category_mappings_foot_5_to_1"     
[17] "category_mappings_foot_5_to_2"      "category_mappings_foot_5_to_3"     
[19] "category_mappings_foot_5_to_4"      "category_mappings_hands_2_to_1"    
[21] "category_mappings_hands_3_to_1"     "category_mappings_hands_3_to_2"    
[23] "category_mappings_hands_4_to_1"     "category_mappings_hands_4_to_2"    
[25] "category_mappings_hands_4_to_3"     "category_

In [2]:
# Create NewSiteData_sum and NewSiteData_ind datasets
n_samples <- 1
NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 24
NewSiteData_sum <- add_element_count(manipulate_rows(NewSiteData_ind))

NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 7
NewSiteData_ind <- add_element_count(manipulate_rows(NewSiteData_ind))

# Save NewSiteData_sum and NewSiteData_ind as .rd files
save(NewSiteData_sum, file = "NewSiteData_sum.rda")
save(NewSiteData_ind, file = "NewSiteData_ind.rda")

print("### SUM ###")
print(colnames(NewSiteData_sum))

[1] "### SUM ###"
 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                
[1] "### SUM ###"
 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                


In [3]:
# Load the datasets
load("HomininPartAbundance_small.rda")
load("HomininPartAbundance_large.rda")
#load("HomininPartAbundance_XL.rda")
load("NewSiteData_sum.rda")
load("NewSiteData_ind.rda")

# If they are already tibbles, you don't need to convert them again.
# You can just start using them as they are
#print(HomininPartAbundance_small)
print("### SUM ###")
print(NewSiteData_sum)
print("### IND ###")
print(NewSiteData_ind)

[1] "### SUM ###"
# A tibble: 1 × 14
  site_index sample_index site_name   sum time_range   geological_context
       <int> <chr>        <chr>     <dbl> <chr>        <chr>             
1          1 11           Site_1        1 Chalcolithic Open air          
# ℹ 8 more variables: number_of_individuals <dbl>, age <chr>, gender <chr>,
#   accumulation_cause <chr>, hominin_species <chr>, bone.level <list>,
#   element <list>, count <list>
[1] "### IND ###"
# A tibble: 7 × 14
  site_index sample_index site_name   sum time_range geological_context
       <int> <chr>        <chr>     <dbl> <chr>      <chr>             
1          1 1_1          Site_1        0 Natufian   Sepulchral Cave   
2          1 1_2          Site_1        0 Natufian   Sepulchral Cave   
3          1 1_3          Site_1        0 Natufian   Sepulchral Cave   
4          1 1_4          Site_1        0 Natufian   Sepulchral Cave   
5          1 1_5          Site_1        0 Natufian   Sepulchral Cave   
6          1 1_6   

In [4]:
result1 <- data.frame(
  bone.level = c(4,4,4,4,4,4, 2,3,3,4))
result2 <- data.frame(
  bone.level = c(1, 2, 3, 3, 4, 4, 4, 4, 4, 4))
result3 <- data.frame(
  bone.level = c(2,5,2,3,1,3, 1,2,3,4))

  min_bone_level <- compareBoneLevels(result1, result2, result3, NewSiteData_sum, NewSiteData_ind[1,],NewSiteData_ind[5,])
  print(min_bone_level)
print(unlist(NewSiteData_sum$bone.level, use.names = FALSE))

 [1] 1 1 2 1 1 1 1 1 1 1
 [1] 3 1 2 1 1 1 2 3 1 3
 [1] 1 1 2 1 1 1 1 1 1 1
 [1] 3 1 2 1 1 1 2 3 1 3


In [5]:
current_mne <- NewSiteData_sum$count[[1]]
bone_names <- NewSiteData_sum$element[[1]]
current_levels <- unlist(NewSiteData_sum$bone.level, use.names = FALSE)
target_levels <-  min_bone_level
sections_order <- c("Skull", "Hyoid", "Vertebrae", "Thorax", "Shoulder",
"Arms", "Hands", "Pelvic", "Legs", "Foot")

In [6]:
source("meta.R")
library(dplyr)

#for (i in 1:5) {
elements_data = elements_data
#current_levels_n = unlist(generate_dataset(n_samples)$bone.level[[1]], use.names =FALSE)
input_df <- get_bones_dataframe(current_levels)
#print(input_df$element)
cl_inp <- summarize_bone_sections(input_df, elements_data)
new_input_df <- get_bones_dataframe(cl_inp)$element
print(identical(input_df$element,new_input_df))
#print(input_df$element)
#print(new_input_df)
print(current_levels)
print(cl_inp)
# Convert both columns to character vectors (if they are not already)
col1 <- as.character(input_df$element)
col2 <- as.character(new_input_df)

# Check if the columns are identical
if (identical(col1, col2)) {
  print("The columns are identical.")
} else {
  print("The columns are not identical.")
  
  # Optionally, you can check for specific differences:
  print("Differences between the two columns:")
  differences <- which(col1 != col2)
  print(data.frame(Row = differences, input_df_element = col1[differences], get_bones_dataframe_element = col2[differences]))
}

[1] TRUE
 [1] 3 1 2 1 1 1 2 3 1 3
 [1] 3 1 2 1 1 1 2 3 1 3
[1] "The columns are identical."
[1] TRUE
 [1] 3 1 2 1 1 1 2 3 1 3
 [1] 3 1 2 1 1 1 2 3 1 3
[1] "The columns are identical."


In [7]:
source("meta.R")
# Assuming you have defined the necessary variables:
# current_mne, current_levels, target_levels, skeleton_hierarchy, bone_names, sections_order, category_hierarchy

# Call the function
new_mne_results <- recal_mne(
  current_mne = current_mne,
  current_levels = current_levels,
  target_levels = target_levels,
  skeleton_hierarchy = skeleton_hierarchy,
  bone_names = bone_names,
  sections_order = sections_order
)

# View the recalculated MNE counts
print(current_levels)
print(target_levels)
print(new_mne_results)

 [1] 3 1 2 1 1 1 2 3 1 3
 [1] 1 1 2 1 1 1 1 1 1 1
# A tibble: 12 × 2
   element            count
   <chr>              <dbl>
 1 Skull_Skull           20
 2 Hyoid_Hyoid            0
 3 Vertebrae_Cervical     2
 4 Vertebrae_Thoracic     9
 5 Vertebrae_Lumbar       1
 6 Thorax_Thorax         18
 7 Shoulder_Shoulder      2
 8 Arms_Arms              4
 9 Hands_Hands           37
10 Pelvic_Pelvic          5
11 Legs_Legs              4
12 Foot_Foot             23
 [1] 3 1 2 1 1 1 2 3 1 3
 [1] 1 1 2 1 1 1 1 1 1 1
# A tibble: 12 × 2
   element            count
   <chr>              <dbl>
 1 Skull_Skull           20
 2 Hyoid_Hyoid            0
 3 Vertebrae_Cervical     2
 4 Vertebrae_Thoracic     9
 5 Vertebrae_Lumbar       1
 6 Thorax_Thorax         18
 7 Shoulder_Shoulder      2
 8 Arms_Arms              4
 9 Hands_Hands           37
10 Pelvic_Pelvic          5
11 Legs_Legs              4
12 Foot_Foot             23


### SELECT

In [ ]:
source("select_data.R")
selected_data <- select_data(
  data = HomininPartAbundance_large,
  #time_range =  "Middle Bronze Age",
  #geological_context = "Rockshelter",
  #number_of_individuals = 70,
  #age = "Adult (17+ years)",
  #gender = "Female",
  #accumulation_cause = "Cannibalism",
  #bone_parts = c("Skull", "Arms"),
  #bone_level = c(2,1,3,4,2,1),
  #site_name = c("Qafzeh", "Regourdou"),
  sum = 1,
  #hominin_species = c("Homo heidelbergensis", "Australopithecus afarensis")
)
colnames(selected_data)

In [ ]:
source("compareBoneLevels.R")
selected_data$bone.level

In [ ]:
# Ensure necessary libraries are loaded
library(dplyr)
library(tidyr)

get_min <- function(selected_data) {

# Convert the 'bone.level' list-column to a data frame
bone_levels_df <- selected_data %>%
  # Select only the 'bone.level' column to simplify
  select(bone.level) %>%
  # Unnest the 'bone.level' list into wider format
  unnest_wider(bone.level)

# Calculate the minimum for each bone section
min_bone_level <- bone_levels_df %>%
  summarise(across(everything(), ~ min(.x, na.rm = TRUE)))

# Convert to a named vector for easier use
min_bone_level_vec <- as.numeric(min_bone_level)
names(min_bone_level_vec) <- colnames(min_bone_level)

# Display the result
return (unname(min_bone_level_vec))
}
get_bones_dataframe(get_min(selected_data))


In [ ]:
select_data_ <- function(
  data,
  time_range_filter = NULL,
  geological_context_filter = NULL,
  number_of_individuals_filter = NULL,
  age_filter = NULL,
  gender_filter = NULL,
  accumulation_cause_filter = NULL,
  bone_parts_filter = NULL,
  site_name_filter = NULL,
  sum_filter = NULL,
  hominin_species_filter = NULL
) {
  library(dplyr)
  library(tidyr)
  library(purrr)

  data_filtered <- data

  # Filter based on metadata fields
  if (!is.null(time_range_filter)) {
    data_filtered <- data_filtered %>%
      filter(time_range %in% time_range_filter)
  }

  if (!is.null(geological_context_filter)) {
    data_filtered <- data_filtered %>%
      filter(geological_context %in% geological_context_filter)
  }

  if (!is.null(number_of_individuals_filter)) {
    if (length(number_of_individuals_filter) == 1) {
      data_filtered <- data_filtered %>%
        filter(number_of_individuals == number_of_individuals_filter)
    } else {
      data_filtered <- data_filtered %>%
        filter(number_of_individuals >= min(number_of_individuals_filter),
               number_of_individuals <= max(number_of_individuals_filter))
    }
  }

  if (!is.null(age_filter)) {
    data_filtered <- data_filtered %>%
      filter(age %in% age_filter)
  }

  if (!is.null(gender_filter)) {
    data_filtered <- data_filtered %>%
      filter(gender %in% gender_filter)
  }

  if (!is.null(accumulation_cause_filter)) {
    data_filtered <- data_filtered %>%
      filter(accumulation_cause %in% accumulation_cause_filter)
  }

  if (!is.null(site_name_filter)) {
    data_filtered <- data_filtered %>%
      filter(site_name %in% site_name_filter)
  }

  if (!is.null(sum_filter)) {
    data_filtered <- data_filtered %>%
      filter(sum %in% sum_filter)
  }

  if (!is.null(hominin_species_filter)) {
    if ("hominin_species" %in% colnames(data_filtered)) {
      data_filtered <- data_filtered %>%
        filter(hominin_species %in% hominin_species_filter)
    } else {
      warning("The 'hominin_species' field is not present in the data.")
    }
  }

  # Filter based on 'element' without unnesting 'bone.level'
  if (!is.null(bone_parts_filter)) {
    data_filtered <- data_filtered %>%
      rowwise() %>%
      filter(any(element %in% bone_parts_filter)) %>%
      ungroup()
  }

  #### Integration of recal_mne function ####

  # Calculate the minimum bone level across the filtered data
  min_bone_level <- get_min(data_filtered)

  # Recalculate 'bone.level', 'element', and 'count' for each row
  data_filtered <- data_filtered %>%
    rowwise() %>%
    mutate(
      # Update bone.level to the minimum bone level
      bone.level = list(min_bone_level),
      # Recalculate counts and elements using recal_mne
      recalculated = list(
        recal_mne(
          current_mne = unlist(count),
          current_levels = unlist(bone.level),
          target_levels = min_bone_level,
          skeleton_hierarchy = skeleton_hierarchy,
          bone_names = unlist(element),
          sections_order = sections_order
        )
      ),
      # Update 'element' and 'count' with recalculated values
      element = list(recalculated$element),
      count = list(recalculated$count)
    ) %>%
    select(-recalculated) %>%
    ungroup()

  return(data_filtered)
}


In [10]:
source("select_data.R")
selected_data_3 <- select_data(
  data = HomininPartAbundance_large,
  time_range =  "Middle Bronze Age",
  geological_context = "Rockshelter",
  #number_of_individuals = 70,
  #age = "Adult (17+ years)",
  gender = "Female",
  #accumulation_cause = "Cannibalism",
  #bone_parts = c("Skull", "Arms"),
  #bone_level = c(2,1,3,4,2,1),
  #site_name = c("Qafzeh", "Regourdou"),
  sum = 1,
  #hominin_species = c("Homo heidelbergensis", "Australopithecus afarensis")
)
colnames(selected_data)

: Error:
! object 'selected_data' not found

In [11]:
print(selected_data %>% select(bone.level, element,count  ))

: Error:
! object 'selected_data' not found

### SELECT 2

In [19]:
source("select_data.R")
selected_data <- select_data(
  data = HomininPartAbundance_large,
  time_range =  "Middle Bronze Age",
  #geological_context = "Cave",
  #number_of_individuals = 70,
  #age = "Adult (17+ years)",
  gender = "Female",
  accumulation_cause = "Cannibalism",
  #bone_parts = c("Skull", "Arms"),
  #bone_level = c(2,1,3,4,2,1),
  #site_name = c("Qafzeh", "Regourdou"),
  sum = 1,
  #hominin_species = c("Homo heidelbergensis", "Australopithecus afarensis")
)
print(colnames(selected_data))

 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                
 [1] "site_index"            "sample_index"          "site_name"            
 [4] "sum"                   "time_range"            "geological_context"   
 [7] "number_of_individuals" "age"                   "gender"               
[10] "accumulation_cause"    "hominin_species"       "bone.level"           
[13] "element"               "count"                


In [20]:
print(selected_data %>% select(bone.level, element,count  ))

# A tibble: 7 × 3
  bone.level        element     count      
  <list>            <list>      <list>     
1 <named list [10]> <chr [59]>  <dbl [59]> 
2 <named list [10]> <chr [102]> <dbl [102]>
3 <named list [10]> <chr [83]>  <dbl [83]> 
4 <named list [10]> <chr [74]>  <dbl [74]> 
5 <named list [10]> <chr [119]> <dbl [119]>
6 <named list [10]> <chr [57]>  <dbl [57]> 
7 <named list [10]> <chr [86]>  <dbl [86]> 
# A tibble: 7 × 3
  bone.level        element     count      
  <list>            <list>      <list>     
1 <named list [10]> <chr [59]>  <dbl [59]> 
2 <named list [10]> <chr [102]> <dbl [102]>
3 <named list [10]> <chr [83]>  <dbl [83]> 
4 <named list [10]> <chr [74]>  <dbl [74]> 
5 <named list [10]> <chr [119]> <dbl [119]>
6 <named list [10]> <chr [57]>  <dbl [57]> 
7 <named list [10]> <chr [86]>  <dbl [86]> 


In [21]:
selected_data$bone.level

[[1]]
[[1]]$Skull
[1] 5

[[1]]$Hyoid
[1] 1

[[1]]$Vertebrae
[1] 1

[[1]]$Thorax
[1] 3

[[1]]$Shoulder
[1] 1

[[1]]$Arms
[1] 1

[[1]]$Hands
[1] 3

[[1]]$Pelvic
[1] 3

[[1]]$Leg
[1] 3

[[1]]$Foot
[1] 2


[[2]]
[[2]]$Skull
[1] 5

[[2]]$Hyoid
[1] 1

[[2]]$Vertebrae
[1] 2

[[2]]$Thorax
[1] 2

[[2]]$Shoulder
[1] 1

[[2]]$Arms
[1] 2

[[2]]$Hands
[1] 3

[[2]]$Pelvic
[1] 2

[[2]]$Leg
[1] 3

[[2]]$Foot
[1] 5


[[3]]
[[3]]$Skull
[1] 3

[[3]]$Hyoid
[1] 1

[[3]]$Vertebrae
[1] 1

[[3]]$Thorax
[1] 4

[[3]]$Shoulder
[1] 1

[[3]]$Arms
[1] 1

[[3]]$Hands
[1] 1

[[3]]$Pelvic
[1] 4

[[3]]$Leg
[1] 2

[[3]]$Foot
[1] 4


[[4]]
[[4]]$Skull
[1] 2

[[4]]$Hyoid
[1] 1

[[4]]$Vertebrae
[1] 2

[[4]]$Thorax
[1] 1

[[4]]$Shoulder
[1] 3

[[4]]$Arms
[1] 1

[[4]]$Hands
[1] 1

[[4]]$Pelvic
[1] 1

[[4]]$Leg
[1] 3

[[4]]$Foot
[1] 5


[[5]]
[[5]]$Skull
[1] 5

[[5]]$Hyoid
[1] 1

[[5]]$Vertebrae
[1] 1

[[5]]$Thorax
[1] 4

[[5]]$Shoulder
[1] 3

[[5]]$Arms
[1] 1

[[5]]$Hands
[1] 2

[[5]]$Pelvic
[1] 1

[[5]]$Leg
[1] 3

[[5]]$Foo

In [30]:
source("recalMNE.R")
  

  
  # Recalculate counts and elements for each row
  recalculated_list <- pmap(
    list(selected_data$count, selected_data$bone.level, selected_data$element),
    function(count_i, bone_level_i, element_i) {
      recal_mne(
        current_mne = count_i,
        current_levels = bone_level_i,
        target_levels = min_bone_level,
        skeleton_hierarchy = skeleton_hierarchy,
        bone_names = element_i,
        sections_order = sections_order
      )
    }
  )

# A tibble: 0 × 6
# ℹ 6 variables: element <chr>, count <dbl>, Level <dbl>,
# A tibble: 0 × 6
# ℹ 6 variables: element <chr>, count <dbl>, Level <dbl>,


#   Skeleton_Section <chr>, std_element <chr>, target_category <chr>
# A tibble: 3 × 6
  element    count Level Skeleton_Section std_element target_category
  <chr>      <dbl> <dbl> <chr>            <chr>       <chr>          
1 Sternum        0     4 Thorax           Sternum     Thorax         
2 Ribs Left      6     3 Thorax           Ribs.Left   Thorax         
3 Ribs Right    11     3 Thorax           Ribs.Right  Thorax         
# A tibble: 6 × 6
  element               count Level Skeleton_Section std_element target_category
  <chr>                 <dbl> <dbl> <chr>            <chr>       <chr>          
1 Hand Carpal Right         5     3 Hands            Hand.Carpa… Hands          
2 Hand Carpal Left          2     3 Hands            Hand.Carpa… Hands          
3 Hand Metacarpal Right     4     3 Hands            Hand.Metac… Hands          
4 Hand Metacarpal Left      4     3 Hands            Hand.Metac… Hands          
5 Hand Phalanx Right        8     3 Hands            Hand.P

   element count Level Skeleton_Section std_element target_category
   <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
 1 S1          0     4 Pelvic           S1          Pelvic         
 2 S2          0     4 Pelvic           S2          Pelvic         
 3 S3          0     4 Pelvic           S3          Pelvic         
 4 S4          1     4 Pelvic           S4          Pelvic         
 5 S5          1     4 Pelvic           S5          Pelvic         
 6 Cx1         0     4 Pelvic           Cx1         Pelvic         
 7 Cx2         0     4 Pelvic           Cx2         Pelvic         
 8 Cx3         0     4 Pelvic           Cx3         Pelvic         
 9 Cx4         1     4 Pelvic           Cx4         Pelvic         
10 Ilium       1     3 Pelvic           Ilium       Pelvic         
11 Ischium     1     3 Pelvic           Ischium     Pelvic         
12 Pubis       1     3 Pelvic           Pubis       Pelvic         
# A tibble: 8 × 6
  element       count Level Sk

# A tibble: 0 × 6
# ℹ 6 variables: element <chr>, count <dbl>, Level <dbl>,
#   Skeleton_Section <chr>, std_element <chr>, target_category <chr>
# A tibble: 3 × 6
  element  count Level Skeleton_Section std_element target_category
  <chr>    <dbl> <dbl> <chr>            <chr>       <chr>          
1 Cervical     6     2 Vertebrae        Cervical    Vertebrae      
2 Thoracic     4     2 Vertebrae        Thoracic    Vertebrae      
3 Lumbar       2     2 Vertebrae        Lumbar      Vertebrae      
# A tibble: 2 × 6
  element count Level Skeleton_Section std_element target_category
  <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
1 Sternum     0     4 Thorax           Sternum     Thorax         
2 Ribs        3     2 Thorax           Ribs        Thorax         
# A tibble: 3 × 6
# A tibble: 0 × 6
# ℹ 6 variables: element <chr>, count <dbl>, Level <dbl>,
#   Skeleton_Section <chr>, std_element <chr>, target_category <chr>
# A tibble: 3 × 6
  element  count Level Skeleto

  element count Level Skeleton_Section std_element target_category
  <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
1 Humerus     1     2 Arms             Humerus     Arms           
2 Radius      0     2 Arms             Radius      Arms           
3 Ulna        2     2 Arms             Ulna        Arms           
# A tibble: 6 × 6
  element               count Level Skeleton_Section std_element target_category
  <chr>                 <dbl> <dbl> <chr>            <chr>       <chr>          
1 Hand Carpal Right         3     3 Hands            Hand.Carpa… Hands          
2 Hand Carpal Left          4     3 Hands            Hand.Carpa… Hands          
3 Hand Metacarpal Right     1     3 Hands            Hand.Metac… Hands          
4 Hand Metacarpal Left      2     3 Hands            Hand.Metac… Hands          
5 Hand Phalanx Right        4     3 Hands            Hand.Phala… Hands          
6 Hand Phalanx Left         5     3 Hands            Hand.Phala… Hands          

...

# A tibble: 2 × 6
  element count Level Skeleton_Section std_element target_category
  <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
1 Sternum     0     4 Thorax           Sternum     Thorax         
2 Ribs       10     2 Thorax           Ribs        Thorax         
# A tibble: 2 × 6
  element  count Level Skeleton_Section std_element target_category
  <chr>    <dbl> <dbl> <chr>            <chr>       <chr>          
1 Clavicle     1     2 Shoulder         Clavicle    Shoulder       
2 Scapula      1     2 Shoulder         Scapula     Shoulder       
# A tibble: 3 × 6
  element count Level Skeleton_Section std_element target_category
  <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
1 Humerus     1     2 Arms             Humerus     Arms           
2 Radius      0     2 Arms             Radius      Arms           
3 Ulna        1     2 Arms             Ulna        Arms           
# A tibble: 32 × 6
   element              count Level Skeleton_Section

   <chr>                <dbl> <dbl> <chr>            <chr>       <chr>          
 1 Hand Trapezium Right     1     5 Hands            Hand.Trape… Hands          
 2 Hand Trapezium Left      1     5 Hands            Hand.Trape… Hands          
 3 Hand Trapezoid Right     0     5 Hands            Hand.Trape… Hands          
 4 Hand Trapezoid Left      0     5 Hands            Hand.Trape… Hands          
 5 Hand Hamate Right        1     5 Hands            Hand.Hamat… Hands          
 6 Hand Hamate Left         1     5 Hands            Hand.Hamat… Hands          
 7 Hand Triquetrum Rig…     0     5 Hands            Hand.Triqu… Hands          
 8 Hand Triquetrum Left     0     5 Hands            Hand.Triqu… Hands          
 9 Hand Lunate Right        0     5 Hands            Hand.Lunat… Hands          
10 Hand Lunate Left         0     5 Hands            Hand.Lunat… Hands          
# ℹ 22 more rows
# ℹ Use `print(n = ...)` to see more rows
# A tibble: 12 × 6
   element count Level Skeleton

# A tibble: 24 × 6
   element count Level Skeleton_Section std_element target_category
   <chr>   <dbl> <dbl> <chr>            <chr>       <chr>          
 1 L1          1     3 Vertebrae        L1          Vertebrae      
 2 L2          0     3 Vertebrae        L2          Vertebrae      
 3 L3          1     3 Vertebrae        L3          Vertebrae      
 4 L4          0     3 Vertebrae        L4          Vertebrae      
 5 L5          0     3 Vertebrae        L5          Vertebrae      
 6 T1          0     3 Vertebrae        T1          Vertebrae      
 7 T2          1     3 Vertebrae        T2          Vertebrae      
 8 T3          1     3 Vertebrae        T3          Vertebrae      
 9 T4          1     3 Vertebrae        T4          Vertebrae      
10 T5          1     3 Vertebrae        T5          Vertebrae      
# ℹ 14 more rows
# ℹ Use `print(n = ...)` to see more rows
# A tibble: 3 × 6
  element    count Level Skeleton_Section std_element target_category
  <chr>      <dbl>

  element        count Level Skeleton_Section std_element    target_category
  <chr>          <dbl> <dbl> <chr>            <chr>          <chr>          
1 Clavicle Right     1     3 Shoulder         Clavicle.Right Shoulder       
2 Clavicle Left      0     3 Shoulder         Clavicle.Left  Shoulder       
3 Scapula Right      1     3 Shoulder         Scapula.Right  Shoulder       
4 Scapula Left       1     3 Shoulder         Scapula.Left   Shoulder       
# A tibble: 6 × 6
  element       count Level Skeleton_Section std_element   target_category
  <chr>         <dbl> <dbl> <chr>            <chr>         <chr>          
1 Humerus Right     1     3 Arms             Humerus.Right Arms           
2 Humerus Left      1     3 Arms             Humerus.Left  Arms           
3 Radius Right      1     3 Arms             Radius.Right  Arms           
4 Radius Left       1     3 Arms             Radius.Left   Arms           
5 Ulna Right        1     3 Arms             Ulna.Right    Arms       

# A tibble: 8 × 6
  element       count Level Skeleton_Section std_element   target_category
  <chr>         <dbl> <dbl> <chr>            <chr>         <chr>          
1 Femur Left        0     3 Legs             Femur.Left    Legs           
2 Femur Right       1     3 Legs             Femur.Right   Legs           
3 Tibia Left        0     3 Legs             Tibia.Left    Legs           
4 Tibia Right       1     3 Legs             Tibia.Right   Legs           
5 Fibula Left       0     3 Legs             Fibula.Left   Legs           
6 Fibula Right      0     3 Legs             Fibula.Right  Legs           
7 Patella Left      0     3 Legs             Patella.Left  Legs           
8 Patella Right     0     3 Legs             Patella.Right Legs           
# A tibble: 30 × 6
   element              count Level Skeleton_Section std_element target_category
   <chr>                <dbl> <dbl> <chr>            <chr>       <chr>          
 1 Foot Talus Right         0     5 Foot           

In [28]:
recalculated_list

[[1]]
# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             0
2 Vertebrae_Vertebrae     1
3 Thorax_Thorax          17
4 Shoulder_Shoulder       1
5 Arms_Arms               2
6 Hands_Hands            28
7 Pelvic_Pelvic           6
8 Legs_Legs               5
9 Foot_Foot              31

[[2]]
# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             1
2 Vertebrae_Vertebrae    12
3 Thorax_Thorax           3
4 Shoulder_Shoulder       2
5 Arms_Arms               3
6 Hands_Hands            19
7 Pelvic_Pelvic           3
8 Legs_Legs               3
9 Foot_Foot              24

[[3]]
# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             8
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             0
 4 Vertebrae_Vertebrae     5
 5 Thorax_Thorax          12
 6 Shoulder_Shoulder       2
 7 Arms_Arms               3
 8 Hands_Hands             0
 9 Pelvic_Pelvic   

In [29]:
selected_data$element

[[1]]
 [1] "Neurocranium_bone.Frontal bone"          
 [2] "Neurocranium_bone.Parietal bone Right"   
 [3] "Neurocranium_bone.Parietal bone Left"    
 [4] "Neurocranium_bone.Temporal Right"        
 [5] "Neurocranium_bone.Temporal Left"         
 [6] "Neurocranium_bone.Occipital"             
 [7] "Neurocranium_bone.Sphenoid"              
 [8] "Neurocranium_bone.Ethmoid"               
 [9] "Schplancnocranium.Maxilla"               
[10] "Schplancnocranium.Palatine bone"         
[11] "Schplancnocranium.Zygomatic bone Right"  
[12] "Schplancnocranium.Zygomatic bone Left"   
[13] "Schplancnocranium.Nasal bone"            
[14] "Schplancnocranium.Lacrimal bone"         
[15] "Schplancnocranium.Vomer bone"            
[16] "Schplancnocranium.Inferior nasal concha" 
[17] "Ossicles_of_the_Middle_Ear.Malleus Right"
[18] "Ossicles_of_the_Middle_Ear.Malleus Left" 
[19] "Ossicles_of_the_Middle_Ear.Incus Right"  
[20] "Ossicles_of_the_Middle_Ear.Incus Left"   
[21] "Ossicles_of_the_Middle_Ear.S

In [24]:
min_bone_level <- get_min(selected_data)
min_bone_level


 [1] 2 1 1 1 1 1 1 1 1 1

In [ ]:
length(recalculated_list[[3]]$element)

In [ ]:
recalculated_list[[3]]$element

In [ ]:
recalculated_list[[2]]$element

In [ ]:
sum(selected_data[2,]$count[[1]])

In [ ]:
selected_data[2,]$element[[1]]

In [25]:
source("recalMNE.R")

for (i in 1:nrow(selected_data)) {
print(recal_mne(
  current_mne = selected_data[i,]$count[[1]],
  current_levels = selected_data[i,]$bone.level[[1]],
  target_levels = min_bone_level,
  skeleton_hierarchy = skeleton_hierarchy,
  bone_names = selected_data[i,]$element[[1]],
  sections_order = sections_order))
}

# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             0
2 Vertebrae_Vertebrae     1
3 Thorax_Thorax          17
4 Shoulder_Shoulder       1
5 Arms_Arms               2
6 Hands_Hands            28
7 Pelvic_Pelvic           6
8 Legs_Legs               5
9 Foot_Foot              31
# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             0
2 Vertebrae_Vertebrae     1
3 Thorax_Thorax          17
4 Shoulder_Shoulder       1
5 Arms_Arms               2
6 Hands_Hands            28
7 Pelvic_Pelvic           6
8 Legs_Legs               5
9 Foot_Foot              31


# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             1
2 Vertebrae_Vertebrae    12
3 Thorax_Thorax           3
4 Shoulder_Shoulder       2
5 Arms_Arms               3
6 Hands_Hands            19
7 Pelvic_Pelvic           3
8 Legs_Legs               3
9 Foot_Foot              24
# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             1
2 Vertebrae_Vertebrae    12
3 Thorax_Thorax           3
4 Shoulder_Shoulder       2
5 Arms_Arms               3
6 Hands_Hands            19
7 Pelvic_Pelvic           3
8 Legs_Legs               3
9 Foot_Foot              24


# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             8
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             0
 4 Vertebrae_Vertebrae     5
 5 Thorax_Thorax          12
 6 Shoulder_Shoulder       2
 7 Arms_Arms               3
 8 Hands_Hands             0
 9 Pelvic_Pelvic           9
10 Legs_Legs               2
11 Foot_Foot              24
# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             8
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             0
 4 Vertebrae_Vertebrae     5
 5 Thorax_Thorax          12
 6 Shoulder_Shoulder       2
 7 Arms_Arms               3
 8 Hands_Hands             0
 9 Pelvic_Pelvic           9
10 Legs_Legs               2
11 Foot_Foot              24


# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             0
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             0
 4 Vertebrae_Vertebrae     3
 5 Thorax_Thorax          11
 6 Shoulder_Shoulder       2
 7 Arms_Arms               5
 8 Hands_Hands            26
 9 Pelvic_Pelvic           4
10 Legs_Legs               2
11 Foot_Foot              31
# A tibble: 9 × 2
  element             count
  <chr>               <dbl>
1 Hyoid_Hyoid             1
2 Vertebrae_Vertebrae    13
3 Thorax_Thorax          13
4 Shoulder_Shoulder       1
5 Arms_Arms               4
6 Hands_Hands            17
7 Pelvic_Pelvic           0
8 Legs_Legs               5
9 Foot_Foot              26
# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             0
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             0
 4 Vertebrae_Vertebrae     3
 5 Thorax_Thorax          11
 6 Shoulder_Shoulder       2
 7 Arms_Arms               

# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             1
 2 Skull_Mandible          0
 3 Hyoid_Hyoid             1
 4 Vertebrae_Vertebrae    14
 5 Thorax_Thorax          10
 6 Shoulder_Shoulder       2
 7 Arms_Arms               2
 8 Hands_Hands            15
 9 Pelvic_Pelvic           9
10 Legs_Legs               3
11 Foot_Foot              34
# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             1
 2 Skull_Mandible          0
 3 Hyoid_Hyoid             1
 4 Vertebrae_Vertebrae    14
 5 Thorax_Thorax          10
 6 Shoulder_Shoulder       2
 7 Arms_Arms               2
 8 Hands_Hands            15
 9 Pelvic_Pelvic           9
10 Legs_Legs               3
11 Foot_Foot              34


# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             9
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             1
 4 Vertebrae_Vertebrae    13
 5 Thorax_Thorax          19
 6 Shoulder_Shoulder       3
 7 Arms_Arms               5
 8 Hands_Hands            16
 9 Pelvic_Pelvic           2
10 Legs_Legs               2
11 Foot_Foot              25
# A tibble: 11 × 2
   element             count
   <chr>               <dbl>
 1 Skull_Skull             9
 2 Skull_Mandible          1
 3 Hyoid_Hyoid             1
 4 Vertebrae_Vertebrae    13
 5 Thorax_Thorax          19
 6 Shoulder_Shoulder       3
 7 Arms_Arms               5
 8 Hands_Hands            16
 9 Pelvic_Pelvic           2
10 Legs_Legs               2
11 Foot_Foot              25


In [ ]:
recalculated_list 